# EDA Visual — Solicitudes de viaje y demanda agregada (zonas calientes)

Inspección **gráfica** de dos niveles de datos:
- **Nivel A** (`solicitudes_profesor.csv`): solicitudes crudas (una fila por viaje).
- **Nivel B** (`demanda_agregada_profesor.csv`): demanda agregada por celda × franja horaria × municipio.

Complementa `eda_tabular.ipynb`. Referencias: `documentation/01-especificacion-dominio-y-esquema.md`, `documentation/03-diccionario-datos.md`.

**Secciones:**
1. Configuración, carga y funciones reutilizables
2. Demanda temporal (A): hora, día de semana, heatmap día×hora
3. Distribución geoespacial (A): por municipio, por zona real, anomalías
4. Mapa de calor de demanda (B): centroide × demand_density por municipio
5. Oferta/demanda (B): histograma supply_demand_ratio; scatter density vs ratio
6. Contexto (A): distancia_km (log), costo_mxn, is_holiday, accepted
7. Correlación (B): heatmap numérico RdBu_r
8. Calidad (A): nulos por columna

## 1. Configuración, carga de datos y funciones reutilizables

In [1]:
import matplotlib
matplotlib.use("Agg")   # backend no interactivo para nbconvert (headless)

from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import seaborn as sns

# Estilo idéntico al del profe
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 110

# Rutas
NOTEBOOK_DIR = Path(".").resolve()
ROOT_DIR = NOTEBOOK_DIR.parent
DATA_DIR = ROOT_DIR / "documentation"
FIG_DIR = ROOT_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)

# Carga Nivel A (solicitudes crudas)
df_a = pd.read_csv(DATA_DIR / "solicitudes_profesor.csv")
df_a["timestamp"] = pd.to_datetime(df_a["timestamp"], errors="coerce")
df_a["is_weekend"] = df_a["is_weekend"].astype(bool)
df_a["is_holiday"] = df_a["is_holiday"].astype(bool)
df_a["accepted"] = df_a["accepted"].astype(bool)

# Carga Nivel B (demanda agregada)
df_b = pd.read_csv(DATA_DIR / "demanda_agregada_profesor.csv")
df_b["is_hot_true"] = df_b["is_hot_true"].astype(bool)

# Bounding boxes por municipio (del config)
BBOX = {
    1: {"lat": (16.70, 16.80), "lng": (-93.16, -93.08)},
    2: {"lat": (16.86, 16.94), "lng": (-92.12, -92.04)},
}

# Nombres de días
DIAS = {0: "Lun", 1: "Mar", 2: "Mié", 3: "Jue", 4: "Vie", 5: "Sáb", 6: "Dom"}

print(f"Nivel A — Registros: {len(df_a):,} | Columnas: {df_a.shape[1]}")
print(f"Nivel B — Registros: {len(df_b):,} | Columnas: {df_b.shape[1]}")
print(f"Figuras guardadas en: {FIG_DIR}")

Nivel A — Registros: 48,783 | Columnas: 14
Nivel B — Registros: 5,357 | Columnas: 14
Figuras guardadas en: C:\Users\eduar\Desktop\Universidad\9VO CUATRI\Integrador\ZonasCalientes-ML\figures


In [2]:
# ──────────────────────────────────────────────────────────────────
# Helper: guardar figura en figures/
# ──────────────────────────────────────────────────────────────────
def _save(fig: plt.Figure, nombre: str) -> None:
    """Guarda `fig` en FIG_DIR/<nombre> con dpi 120 y cierra la figura."""
    ruta = FIG_DIR / nombre
    fig.savefig(ruta, bbox_inches="tight", dpi=120)
    print(f"Guardada: {ruta.name}")
    plt.close(fig)


# ──────────────────────────────────────────────────────────────────
# plot_numerica: histograma + KDE (escala log opcional)
# ──────────────────────────────────────────────────────────────────
def plot_numerica(
    col: str,
    data: pd.DataFrame | None = None,
    hue: str | None = None,
    log_x: bool = False,
    bins: int = 40,
    nombre: str | None = None,
) -> None:
    """Histograma con KDE para una variable numérica continua."""
    src = data if data is not None else df_a
    fig, ax = plt.subplots(figsize=(10, 5))
    serie = src.dropna(subset=[col])
    if hue and hue in src.columns:
        for cat, grp in serie.groupby(hue, observed=True):
            sns.histplot(grp[col], bins=bins, kde=True, stat="density",
                         alpha=0.35, label=str(cat), ax=ax)
        ax.legend(title=hue, bbox_to_anchor=(1.02, 1), loc="upper left")
    else:
        sns.histplot(serie[col], bins=bins, kde=True, color="steelblue", ax=ax)
    if log_x:
        ax.set_xscale("log")
    n_null = src[col].isna().sum()
    ax.set_title(f"{col}  (nulos: {n_null}, {n_null / len(src) * 100:.1f} %)")
    ax.set_xlabel(col)
    plt.tight_layout()
    _save(fig, nombre or f"num_{col}.png")


# ──────────────────────────────────────────────────────────────────
# plot_box: boxplot agrupado
# ──────────────────────────────────────────────────────────────────
def plot_box(
    col: str,
    by: str,
    data: pd.DataFrame | None = None,
    nombre: str | None = None,
) -> None:
    """Boxplot de `col` agrupado por `by`."""
    src = data if data is not None else df_a
    order = (
        src.groupby(by, observed=True)[col]
        .median()
        .sort_values(ascending=False)
        .index
    )
    fig, ax = plt.subplots(figsize=(11, 5))
    sns.boxplot(data=src, x=by, y=col, order=order, ax=ax, fliersize=2)
    ax.set_title(f"{col} por {by}")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    _save(fig, nombre or f"box_{col}_by_{by}.png")


# ──────────────────────────────────────────────────────────────────
# plot_categorica: barras horizontales de value_counts
# ──────────────────────────────────────────────────────────────────
def plot_categorica(
    col: str,
    data: pd.DataFrame | None = None,
    top_n: int | None = None,
    nombre: str | None = None,
) -> None:
    """Barras de frecuencia para variable categórica."""
    src = data if data is not None else df_a
    vc = src[col].value_counts(dropna=False)
    if top_n:
        vc = vc.head(top_n)
    fig, ax = plt.subplots(figsize=(10, max(4, 0.35 * len(vc))))
    sns.barplot(x=vc.values, y=vc.index.astype(str), ax=ax, color="steelblue")
    n_null = src[col].isna().sum()
    ax.set_title(f"{col}  (nulos: {n_null})")
    ax.set_xlabel("conteo")
    plt.tight_layout()
    _save(fig, nombre or f"cat_{col}.png")


# ──────────────────────────────────────────────────────────────────
# plot_bool: barras simples o apiladas por otra variable
# ──────────────────────────────────────────────────────────────────
def plot_bool(
    col: str,
    data: pd.DataFrame | None = None,
    hue: str | None = None,
    nombre: str | None = None,
) -> None:
    """Barras para variable booleana, con desglose opcional por hue."""
    src = data if data is not None else df_a
    fig, ax = plt.subplots(figsize=(7, 4))
    if hue:
        ct = pd.crosstab(src[hue], src[col], dropna=False)
        ct.plot(kind="bar", stacked=True, ax=ax, colormap="Set2")
        ax.legend(title=col, bbox_to_anchor=(1.02, 1))
        ax.set_title(f"{col} por {hue}")
    else:
        vc = src[col].value_counts(dropna=False)
        sns.barplot(x=vc.index.astype(str), y=vc.values, ax=ax, color="steelblue")
        ax.set_title(col)
        ax.set_xlabel(col)
        ax.set_ylabel("conteo")
    plt.tight_layout()
    _save(fig, nombre or f"bool_{col}.png")


print("Funciones reutilizables definidas: _save, plot_numerica, plot_box,")
print("  plot_categorica, plot_bool")

Funciones reutilizables definidas: _save, plot_numerica, plot_box,
  plot_categorica, plot_bool


## 2. Demanda temporal (Nivel A)

Analizamos **cuándo** se concentran las solicitudes de viaje:
- Viajes por **hora del día** (línea).
- Viajes por **día de semana** (barras).
- **Heatmap día de semana × hora** para revelar la doble estacionalidad.

In [3]:
# ── 2a. Viajes por hora (línea) ─────────────────────────────────
# Filtramos horas válidas (algunas anomalías inyectadas tienen hour > 23)
df_a_hora = df_a[df_a["hour"].between(0, 23)]
conteo_hora = df_a_hora.groupby("hour").size().reset_index(name="viajes")

fig, ax = plt.subplots(figsize=(11, 5))
sns.lineplot(
    data=conteo_hora, x="hour", y="viajes",
    marker="o", ax=ax, color="steelblue"
)
ax.set_title("Solicitudes de viaje por hora del día (Nivel A)")
ax.set_xlabel("Hora")
ax.set_ylabel("Número de solicitudes")
ax.set_xticks(range(0, 24))
plt.tight_layout()
_save(fig, "temporal_viajes_por_hora.png")
print("Figura: temporal_viajes_por_hora.png")

Guardada: temporal_viajes_por_hora.png
Figura: temporal_viajes_por_hora.png


In [4]:
# ── 2b. Viajes por día de semana (barras) ────────────────────────
conteo_dia = df_a.groupby("day_of_week").size().reset_index(name="viajes")
conteo_dia["dia_nombre"] = conteo_dia["day_of_week"].map(DIAS)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(
    data=conteo_dia, x="dia_nombre", y="viajes",
    order=[DIAS[i] for i in range(7)],
    ax=ax, color="steelblue"
)
ax.set_title("Solicitudes de viaje por día de semana (Nivel A)")
ax.set_xlabel("Día")
ax.set_ylabel("Número de solicitudes")
plt.tight_layout()
_save(fig, "temporal_viajes_por_dia_semana.png")
print("Figura: temporal_viajes_por_dia_semana.png")

Guardada: temporal_viajes_por_dia_semana.png
Figura: temporal_viajes_por_dia_semana.png


In [5]:
# ── 2c. Heatmap día de semana × hora ─────────────────────────────
# Solo filas con hora válida
pivot = (
    df_a_hora
    .groupby(["day_of_week", "hour"])
    .size()
    .unstack(fill_value=0)
)
pivot.index = [DIAS[i] for i in pivot.index]

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(
    pivot,
    cmap="YlOrRd",
    linewidths=0.3,
    linecolor="white",
    ax=ax,
    cbar_kws={"label": "Número de solicitudes"},
)
ax.set_title("Heatmap de demanda: día de semana × hora (Nivel A)")
ax.set_xlabel("Hora del día")
ax.set_ylabel("Día de semana")
plt.tight_layout()
_save(fig, "temporal_heatmap_dia_hora.png")
print("Figura: temporal_heatmap_dia_hora.png")

Guardada: temporal_heatmap_dia_hora.png


Figura: temporal_heatmap_dia_hora.png


## 3. Distribución geoespacial (Nivel A)

Visualizamos los **orígenes de viaje** en el plano (lng, lat):
- Por **municipio** (dos clusters geográficos distintos).
- Por **zone_id_true** (zona plantada; -1 = ruido uniforme).
- Marcando **anomalías** (coordenadas fuera de la bounding box del municipio).

In [6]:
# ── 3a. Scatter por municipio ────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
paleta_muni = {1: "steelblue", 2: "tomato"}
for muni, grp in df_a.groupby("municipio"):
    ax.scatter(
        grp["lng"], grp["lat"],
        s=4, alpha=0.25,
        color=paleta_muni.get(int(muni), "gray"),
        label=f"Municipio {muni}",
        rasterized=True,
    )
ax.set_title("Orígenes de solicitud por municipio (Nivel A)")
ax.set_xlabel("Longitud")
ax.set_ylabel("Latitud")
ax.legend(title="Municipio", markerscale=4)
plt.tight_layout()
_save(fig, "geo_scatter_por_municipio.png")
print("Figura: geo_scatter_por_municipio.png")

Guardada: geo_scatter_por_municipio.png
Figura: geo_scatter_por_municipio.png


In [7]:
# ── 3b. Scatter por zone_id_true (zonas plantadas) ───────────────
# Paleta: zona -1 (ruido) en gris; zonas 0..N en colores vivos
ZONA_COLORES = {
    -1: "lightgray",
     0: "steelblue",
     1: "tomato",
     2: "seagreen",
     3: "darkorange",
     4: "mediumpurple",
}

fig, ax = plt.subplots(figsize=(11, 7))

# Primero el ruido (gris, detrás)
ruido = df_a[df_a["zone_id_true"] == -1]
ax.scatter(
    ruido["lng"], ruido["lat"],
    s=3, alpha=0.15, color=ZONA_COLORES[-1],
    label="Ruido (-1)", rasterized=True
)

# Luego cada zona real (encima)
for zona in sorted(df_a["zone_id_true"].dropna().unique()):
    zona = int(zona)
    if zona == -1:
        continue
    g = df_a[df_a["zone_id_true"] == zona]
    ax.scatter(
        g["lng"], g["lat"],
        s=5, alpha=0.35,
        color=ZONA_COLORES.get(zona, "purple"),
        label=f"Zona {zona}",
        rasterized=True
    )

ax.set_title("Orígenes de solicitud por zona plantada (ground-truth) — Nivel A")
ax.set_xlabel("Longitud")
ax.set_ylabel("Latitud")
ax.legend(title="zone_id_true", markerscale=4, loc="best")
plt.tight_layout()
_save(fig, "geo_scatter_por_zona_real.png")
print("Figura: geo_scatter_por_zona_real.png")

Guardada: geo_scatter_por_zona_real.png
Figura: geo_scatter_por_zona_real.png


In [8]:
# ── 3c. Scatter con anomalías marcadas (fuera de bbox) ───────────
# Determinamos anomalías: coordenadas fuera del bbox de su municipio
def _es_anomalia(row) -> bool:
    bb = BBOX.get(int(row["municipio"]), None)
    if bb is None:
        return False
    return (
        not bb["lat"][0] <= row["lat"] <= bb["lat"][1]
        or not bb["lng"][0] <= row["lng"] <= bb["lng"][1]
    )

df_a_valid = df_a.dropna(subset=["lat", "lng"])
mask_anomalia = df_a_valid.apply(_es_anomalia, axis=1)
normales = df_a_valid[~mask_anomalia]
anomalos = df_a_valid[mask_anomalia]

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(
    normales["lng"], normales["lat"],
    s=4, alpha=0.2, color="steelblue",
    label="Normal", rasterized=True
)
ax.scatter(
    anomalos["lng"], anomalos["lat"],
    s=30, alpha=0.85, color="crimson", marker="x",
    label=f"Anomalía coords (n={len(anomalos):,})", zorder=5
)
ax.set_title("Orígenes de solicitud — anomalías de coordenadas (fuera de bbox)")
ax.set_xlabel("Longitud")
ax.set_ylabel("Latitud")
ax.legend(markerscale=2)
plt.tight_layout()
_save(fig, "geo_scatter_anomalias.png")
print(f"Figura: geo_scatter_anomalias.png  (anomalías: {len(anomalos):,})")

Guardada: geo_scatter_anomalias.png
Figura: geo_scatter_anomalias.png  (anomalías: 490)


## 4. Mapa de calor de demanda (Nivel B)

Scatter de centroides de celda (`centroid_lon`, `centroid_lat`),
con **tamaño y color proporcional a `demand_density`** (solicitudes/km²).
Las **celdas calientes** (`is_hot_true=True`) deben saltar a la vista.
Graficamos un panel por municipio para apreciar la estructura espacial.

In [9]:
# ── 4. Mapa de calor de demanda (Nivel B) ────────────────────────
munis = sorted(df_b["municipio"].unique())
n_munis = len(munis)

fig, axes = plt.subplots(1, n_munis, figsize=(7 * n_munis, 6))
if n_munis == 1:
    axes = [axes]

# Normalizar demand_density globalmente para comparar municipios
d_min = df_b["demand_density"].min()
d_max = df_b["demand_density"].max()
norm = mcolors.Normalize(vmin=d_min, vmax=d_max)
cmap = plt.get_cmap("YlOrRd")

for ax, muni in zip(axes, munis):
    sub = df_b[df_b["municipio"] == muni].copy()

    # Tamaño proporcional a demand_density (normalizado a 5–200)
    d_range = sub["demand_density"].max() - sub["demand_density"].min() + 1e-9
    sizes = 5 + 195 * (sub["demand_density"] - sub["demand_density"].min()) / d_range

    sc = ax.scatter(
        sub["centroid_lon"], sub["centroid_lat"],
        c=sub["demand_density"],
        s=sizes,
        cmap=cmap,
        norm=norm,
        alpha=0.75,
        edgecolors="none",
        rasterized=True,
    )

    # Resaltar celdas HOT con borde negro
    hot = sub[sub["is_hot_true"]]
    hot_sizes = 5 + 195 * (hot["demand_density"] - sub["demand_density"].min()) / d_range
    ax.scatter(
        hot["centroid_lon"], hot["centroid_lat"],
        c=hot["demand_density"],
        s=hot_sizes * 1.3,
        cmap=cmap,
        norm=norm,
        alpha=0.95,
        edgecolors="black",
        linewidths=0.6,
        rasterized=True,
        label=f"Hot (n={len(hot):,})",
    )

    ax.set_title(f"Municipio {muni} — demand\_density\n(celdas hot con borde negro)")
    ax.set_xlabel("Longitud")
    ax.set_ylabel("Latitud")
    ax.legend(loc="upper right", fontsize=8)

# Colorbar compartida
cbar = fig.colorbar(sc, ax=axes, shrink=0.8, pad=0.02)
cbar.set_label("demand_density (solicitudes/km²)")
fig.suptitle("Mapa de calor de demanda — celdas × franja horaria (Nivel B)", fontsize=13)
plt.tight_layout()
_save(fig, "demanda_mapa_calor_nivel_b.png")
print("Figura: demanda_mapa_calor_nivel_b.png")

C:\Users\eduar\AppData\Local\Temp\ipykernel_47068\3205393724.py:58: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


Guardada: demanda_mapa_calor_nivel_b.png
Figura: demanda_mapa_calor_nivel_b.png


## 5. Oferta / demanda (Nivel B)

Exploramos la relación entre oferta y demanda:
- **Histograma** de `supply_demand_ratio` (conductores / solicitudes).
- **Scatter** `demand_density` vs `supply_demand_ratio` coloreado por `is_hot_true`
  para ver si las zonas calientes se separan claramente.

In [10]:
# ── 5a. Histograma supply_demand_ratio ───────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(
    df_b["supply_demand_ratio"],
    bins=50, kde=True,
    color="steelblue", ax=ax
)
ax.axvline(
    x=1.0, color="crimson", linestyle="--", linewidth=1.5,
    label="ratio = 1 (equilibrio oferta/demanda)"
)
ax.set_title("Distribución del supply_demand_ratio (Nivel B)\n"
             "(< 1 → demanda supera oferta; > 1 → oferta supera demanda)")
ax.set_xlabel("supply_demand_ratio")
ax.set_ylabel("conteo")
ax.legend()
plt.tight_layout()
_save(fig, "oferta_demanda_ratio_hist.png")
print("Figura: oferta_demanda_ratio_hist.png")

Guardada: oferta_demanda_ratio_hist.png
Figura: oferta_demanda_ratio_hist.png


In [11]:
# ── 5b. Scatter demand_density vs supply_demand_ratio ────────────
# Color: is_hot_true (True = caliente, False = fría)
colores_hot = {True: "tomato", False: "steelblue"}
etiquetas_hot = {True: "Hot (is_hot_true=True)", False: "No hot"}

fig, ax = plt.subplots(figsize=(10, 6))

# Primero no-hot (detrás)
for es_hot in [False, True]:
    sub = df_b[df_b["is_hot_true"] == es_hot]
    ax.scatter(
        sub["demand_density"],
        sub["supply_demand_ratio"],
        s=12 if es_hot else 8,
        alpha=0.6 if es_hot else 0.3,
        color=colores_hot[es_hot],
        label=f"{etiquetas_hot[es_hot]} (n={len(sub):,})",
        rasterized=True,
    )

ax.set_xlabel("demand_density (solicitudes/km²)")
ax.set_ylabel("supply_demand_ratio")
ax.set_title("demand_density vs supply_demand_ratio — coloreado por is_hot_true (Nivel B)")
ax.legend(title="Zona caliente", loc="upper right")
# Línea de equilibrio oferta/demanda
ax.axhline(y=1.0, color="gray", linestyle=":", linewidth=1, label="ratio=1")
plt.tight_layout()
_save(fig, "oferta_demanda_scatter_density_ratio.png")
print("Figura: oferta_demanda_scatter_density_ratio.png")

Guardada: oferta_demanda_scatter_density_ratio.png
Figura: oferta_demanda_scatter_density_ratio.png


## 6. Contexto: atributos del viaje (Nivel A)

Distribuciones de variables de contexto:
- `distancia_km` (escala log por la distribución log-normal).
- `costo_mxn` (tarifas fijas por zona).
- `is_holiday` y `accepted` (booleanas).

In [12]:
# ── 6a. Histograma distancia_km (escala log) ─────────────────────
plot_numerica("distancia_km", log_x=True, nombre="ctx_distancia_km_log.png")
print("Figura: ctx_distancia_km_log.png")

Guardada: ctx_distancia_km_log.png
Figura: ctx_distancia_km_log.png


In [13]:
# ── 6b. Histograma costo_mxn ─────────────────────────────────────
# El costo es una tarifa FIJA por zona de destino (discreta)
plot_numerica("costo_mxn", bins=20, nombre="ctx_costo_mxn.png")
print("Figura: ctx_costo_mxn.png")

Guardada: ctx_costo_mxn.png
Figura: ctx_costo_mxn.png


In [14]:
# ── 6c. Bool is_holiday ──────────────────────────────────────────
plot_bool("is_holiday", nombre="ctx_is_holiday.png")
print("Figura: ctx_is_holiday.png")

Guardada: ctx_is_holiday.png
Figura: ctx_is_holiday.png


In [15]:
# ── 6d. Bool accepted (tasa de atención) ─────────────────────────
plot_bool("accepted", nombre="ctx_accepted.png")
print("Figura: ctx_accepted.png")

Guardada: ctx_accepted.png
Figura: ctx_accepted.png


## 7. Heatmap de correlación — variables numéricas (Nivel B)

`cmap=RdBu_r, center=0` para distinguir correlaciones positivas (azul) y negativas (rojo).
Esperamos correlación negativa entre `demand_density` y `supply_demand_ratio`.

In [16]:
# ── 7. Heatmap de correlación (Nivel B) ──────────────────────────
NUM_COLS_B = [
    "hour",
    "n_requests",
    "n_drivers_available",
    "supply_demand_ratio",
    "demand_density",
    "cancel_rate",
]

corr = df_b[NUM_COLS_B].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    corr,
    cmap="RdBu_r",
    center=0,
    vmin=-1, vmax=1,
    annot=True, fmt=".2f",
    square=True,
    linewidths=0.4,
    ax=ax,
    cbar_kws={"label": "Pearson r"},
)
ax.set_title("Matriz de correlación — variables numéricas del Nivel B")
plt.tight_layout()
_save(fig, "corr_heatmap.png")
print("Figura: corr_heatmap.png")

Guardada: corr_heatmap.png
Figura: corr_heatmap.png


## 8. Calidad de datos — nulos por columna (Nivel A)

Barras de porcentaje de nulos en el Nivel A (inyectados aleatoriamente en `distancia_km` y `costo_mxn`).

In [17]:
# ── 8. Nulos por columna (Nivel A) ───────────────────────────────
nulos_pct = (df_a.isna().mean() * 100).sort_values(ascending=True)
nulos_pct = nulos_pct[nulos_pct > 0]

if len(nulos_pct) == 0:
    print("No hay nulos en el Nivel A.")
else:
    fig, ax = plt.subplots(figsize=(10, max(4, 0.5 * len(nulos_pct))))
    sns.barplot(
        x=nulos_pct.values,
        y=nulos_pct.index.astype(str),
        ax=ax, color="coral"
    )
    ax.set_title("Porcentaje de nulos por columna — Nivel A")
    ax.set_xlabel("% nulos")
    # Anotar el valor sobre cada barra
    for p in ax.patches:
        ax.annotate(
            f"{p.get_width():.2f} %",
            (p.get_width() + 0.02, p.get_y() + p.get_height() / 2),
            va="center", fontsize=9,
        )
    plt.tight_layout()
    _save(fig, "calidad_nulos_por_columna.png")
    print("Figura: calidad_nulos_por_columna.png")

print()
print(nulos_pct.rename("% nulos").to_frame().to_string())

Guardada: calidad_nulos_por_columna.png
Figura: calidad_nulos_por_columna.png



               % nulos
distancia_km  1.465675
costo_mxn     1.535371


In [18]:
# ── Resumen final ─────────────────────────────────────────────────
figs = sorted(FIG_DIR.glob("*.png"))
print(f"\nTotal de figuras en figures/: {len(figs)}")
for f in figs:
    print(f"  {f.name}")


Total de figuras en figures/: 18
  calidad_nulos_por_columna.png
  corr_heatmap.png
  ctx_accepted.png
  ctx_costo_mxn.png
  ctx_distancia_km_log.png
  ctx_is_holiday.png
  ctx_lluvia.png
  ctx_pasajeros.png
  ctx_tipo_servicio.png
  demanda_mapa_calor_nivel_b.png
  geo_scatter_anomalias.png
  geo_scatter_por_municipio.png
  geo_scatter_por_zona_real.png
  oferta_demanda_ratio_hist.png
  oferta_demanda_scatter_density_ratio.png
  temporal_heatmap_dia_hora.png
  temporal_viajes_por_dia_semana.png
  temporal_viajes_por_hora.png
